# 🛡️ Sentinel 10-Class Gujarat Police CCTV AI — Kaggle GPU Training Suite
## Trained Directly on 4,050+ Real Gujarat CCTV Video Frames across 30 Police Surveillance Cameras

### 🎯 The 10 Practical Indian Road Classes:
1. `0: pedestrian` — Foot travelers, street crossing citizens, vendors
2. `1: car` — Sedans, hatchbacks, private SUVs, MUVs
3. `2: two_wheeler` — Motorcycles, scooters, mopeds, bicycles (eliminates high-angle confusion)
4. `3: heavy_machinery` — Tractors, JCBs, excavators, cranes, road rollers
5. `4: emergency_vehicle` — Ambulances, police patrol PCRs, fire tenders
6. `5: van` — Maruti Omni, Eeco, Tempo Travelers, commercial delivery vans
7. `6: truck` — Freight lorries, dumpers, multi-axle cargo carriers
8. `7: bus` — State transit GSRTC buses, private luxury coaches, school buses
9. `8: auto_rickshaw` — 3-wheel autos, e-rickshaws, chhakdas
10. `9: others` — Animal/bullock carts, handcarts (thelas), misc transport

---
### 📹 Real Gujarat CCTV Conditions Covered:
- **All 30 Live Gujarat Police Cameras**: Ahmedabad (Chimanbhai Bridge, Visat Junction), Surat, Vadodara, Rajkot, Bilimora, etc.
- **Extreme Lighting Variations**: Morning rush hour, harsh afternoon sun glare, dusk twilight, and nighttime sodium vapor illumination.
- **Surveillance Sensor Profile**: Real H.264 compression artifacts, motion blur, steep high-pole CCTV angles, and dense mixed-traffic occlusion.

In [ ]:
# ─── Cell 1: Hardware & CUDA Verification ─────────────────────────────
!nvidia-smi

import torch
assert torch.cuda.is_available(), "⚠️ GPU is not enabled! In Kaggle, click Settings (right sidebar) -> Accelerator -> select GPU P100 or T4 x2 -> Save."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\n⚡ GPU Active: {gpu_name} ({vram_gb:.2f} GB VRAM) — Ready for 10-Class CCTV Training!")

In [ ]:
# ─── Cell 2: Install Ultralytics & Vision Tools ───────────────────────
!pip install -q ultralytics albumentations pyyaml opencv-python matplotlib seaborn

import ultralytics
print(f"✅ Ultralytics Version: {ultralytics.__version__}")

In [ ]:
# ─── Cell 3: Detect & Prepare 10-Class Gujarat CCTV Dataset ────────────
import os, glob, yaml, zipfile, shutil

# 1. Check if uploaded as a ZIP file (e.g. SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip)
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
working_dataset_dir = "/kaggle/working/sentinel_10class_gujarat_dataset"

if zip_files and not os.path.exists(working_dataset_dir):
    target_zip = zip_files[0]
    print(f"📦 Unzipping uploaded dataset archive: {target_zip}...")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall("/kaggle/working/")
    print("✅ Extraction Complete!")

# 2. Find data.yaml in either /kaggle/working or /kaggle/input
found_yamls = glob.glob("/kaggle/working/**/data.yaml", recursive=True) + glob.glob("/kaggle/input/**/data.yaml", recursive=True)

if found_yamls:
    dataset_yaml = found_yamls[0]
    dataset_root = os.path.dirname(dataset_yaml)
    print(f"\n✅ Found 10-class dataset at: {dataset_root}")
    
    with open(dataset_yaml, 'r') as f:
        data_cfg = yaml.safe_load(f)
        
    # Overwrite dataset path for Kaggle environment
    data_cfg['path'] = dataset_root
    kaggle_yaml_path = "/kaggle/working/data.yaml"
    with open(kaggle_yaml_path, 'w') as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
        
    train_imgs = glob.glob(f"{dataset_root}/images/train/*.*")
    val_imgs = glob.glob(f"{dataset_root}/images/val/*.*")
    print(f"📊 Total Dataset: {len(train_imgs)} Training Frames | {len(val_imgs)} Validation Frames")
    print(f"🏷️ Classes ({len(data_cfg.get('names', []))}): {data_cfg.get('names')}")
else:
    raise FileNotFoundError("⚠️ Dataset not found! Make sure 'SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip' or dataset folder is added under Kaggle Input.")

In [ ]:
# ─── Cell 4: Initialize YOLO Architecture (YOLO11 / YOLOv12 / YOLOv8) ───
from ultralytics import YOLO

# YOLO11s/YOLOv12s provides state-of-the-art accuracy on small, blurred objects with 60+ FPS on edge
MODEL_NAME = "yolo11s.pt"  # or 'yolo12s.pt'
model = YOLO(MODEL_NAME)
print(f"🚀 Loaded base architecture: {MODEL_NAME}")

In [ ]:
# ─── Cell 5: CCTV-Optimized GPU Training (40-50 Epochs) ───────────────
# Augmentation tuned specifically for Indian CCTV conditions:
#   - imgsz=640: Native surveillance camera resolution (fast + robust)
#   - mosaic=1.0: Multi-scale occlusion training (handles traffic jams)
#   - mixup=0.15: Fuses vehicle contours for low-light night sodium vision
#   - hsv_v=0.45: Extreme brightness jitter for headlight glare & shadows
#   - hsv_s=0.60: Robustness to night sodium color shifts

results = model.train(
    data=kaggle_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    # CCTV-Specific Augmentations
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.45,
    degrees=5.0,
    translate=0.1,
    scale=0.35,
    fliplr=0.5,
    project="/kaggle/working/runs",
    name="sentinel_10class_gujarat",
    save=True,
    save_period=10,
    patience=12,
    exist_ok=True
)

print("🎉 Training complete!")

In [ ]:
# ─── Cell 6: Detailed 10-Class Validation & Evaluation ─────────────────
val_results = model.val(data=kaggle_yaml_path, imgsz=640, split="val")

print("\n🏆 Overall Validation Metrics:")
print(f"   • Overall mAP@50:    {val_results.box.map50 * 100:.2f}%")
print(f"   • Overall mAP@50-95: {val_results.box.map * 100:.2f}%")

print("\n📈 Per-Class Breakdown (All 10 Classes):")
for i, cname in model.names.items():
    p = val_results.box.p[i] * 100
    r = val_results.box.r[i] * 100
    m50 = val_results.box.maps[i] * 100
    print(f"   • {cname.ljust(18)} | Precision: {p:5.1f}% | Recall: {r:5.1f}% | mAP50: {m50:5.1f}%")

In [ ]:
# ─── Cell 7: Package & Export Trained Model Weights for Sentinel ──────
best_weights_path = "/kaggle/working/runs/sentinel_10class_gujarat/weights/best.pt"
output_weights = "/kaggle/working/sentinel_10class_traffic_best.pt"

if os.path.exists(best_weights_path):
    shutil.copy2(best_weights_path, output_weights)
    size_mb = os.path.getsize(output_weights) / (1024 * 1024)
    print(f"\n🏆 SUCCESS! High-Accuracy 10-Class Model Ready:")
    print(f"   • Output Weights: {output_weights}")
    print(f"   • File Size:      {size_mb:.2f} MB")
    print("\n⬇️ In Kaggle's right-hand Output sidebar, click '...' next to 'sentinel_10class_traffic_best.pt' -> Download!")
    
    # Export ONNX for high-speed edge deployment
    try:
        model.export(format="onnx", dynamic=True, simplify=True)
        print("✅ Also exported ONNX format into /kaggle/working/!")
    except Exception as e:
        print(f"Note on ONNX export: {e}")
else:
    print("⚠️ Check runs directory for best.pt")